# 10. Local programs

How DSL operators run on the ranks of a distributed run: each rank sees its part of the mesh, a *local mesh*, with its own numbering, provided by `with local_mesh:`; *local programs* run on every rank and call the *collectives* `exchange` and `global_sum`. Here the ranks are threads of one process: a stand-in for MPI.

The example: mass sharing on a plate of 4 × 2 cells, cell `c = 4 j + i` at column `i` and row `j`, and two partitions of it: `halves`, the columns 0-1 and 2-3, and `thirds`, the columns 0, 1-2 and 3. The same computation as a global program, without `Owned` and `exchange` in the source, is in "13. Global programs".

```
  cells               halves (ranks)      thirds (ranks)
  +---+---+---+---+   +---+---+---+---+   +---+---+---+---+
  | 4 | 5 | 6 | 7 |   | 0   0 | 1   1 |   | 0 | 1   1 | 2 |
  +---+---+---+---+   +       +       +   +   +       +   +
  | 0 | 1 | 2 | 3 |   | 0   0 | 1   1 |   | 0 | 1   1 | 2 |
  +---+---+---+---+   +---+---+---+---+   +---+---+---+---+
```

Contents: local meshes and decomposition · domains on a rank · operators on a rank · local programs · the simulation condition · a missing exchange · reproducible sums · fields are values; what DSL operators cannot do.

## Local meshes and decomposition

| Concept | Kind | Meaning |
|---|---|---|
| `LocalMesh` | class (a `Mesh`) | The mesh of one rank, with local indices, built once, checked, immutable: its `tables` (a row per local point, the halo too), `extents`, `owned` points, `global_index` (the global index of each local point), `regions`, `mesh_id` (the same on all the ranks) and `rank` |
| `TRUNCATED` | constant | `-3`: a neighbor outside the local points, only in halo rows. Never read as a value: a read reaching it is not computable |
| `Region[D]` | class | A named set of points of a `Location` `D`, declared by subclassing: `class West(Region[Cell]): ...` |
| `decompose(tables, extents, regions, owner, support=..., hops=..., derived=...)` | function | The `LocalMesh` of each rank of the partition `owner`: the points a rank owns, then its halo, those reached through the relations of `support`, `hops` times, each in global order. The tables are checked (or derived, for stencils) on the global mesh, then localized; `derived` names more stencils to derive globally, e.g. those with transposes, which a local mesh does not derive |

`C2C` gives the cells beside each cell, `[west, east, south, north]`, missing at the boundary of the plate; `Near`, its transpose, lists them in increasing global order.

In [1]:
import contextlib
import io

import numpy as np
from concepts import (
    MISSING_VALUE,
    TRUNCATED,
    Dims,
    Field,
    Location,
    Mesh,
    Neighbor,
    Region,
    Relation,
    Stencil,
)
from implementation import NumPyArrayField, decompose, neighbor_table, sum_over


class Cell(Location): ...


class C2C(Relation[Cell, Cell], max_neighbors=4, partial=True): ...  # [west, east, south, north]


class Near(Relation[Cell, Cell], max_neighbors=4, skip_values=True, stencil=Stencil(C2C.T)): ...


class West(Region[Cell]): ...


class East(Region[Cell]): ...


def at(i, j):  # the cell at column i, row j, or missing
    return 4 * j + i if 0 <= i < 4 and 0 <= j < 2 else MISSING_VALUE


rows = [
    [at(i - 1, j), at(i + 1, j), at(i, j - 1), at(i, j + 1)] for j in range(2) for i in range(4)
]
plate = {C2C: neighbor_table(C2C, rows)}
extents, regions = {Cell: 8}, {West: (0, 4), East: (3, 7)}


def partition(owner, derived=()):
    """The local meshes of the plate, for the owner rank of each cell."""
    return decompose(
        plate, extents, regions, {Cell: owner}, support=(C2C,), hops=1, derived=derived
    )


single = partition([0] * 8)
halves = partition([0, 0, 1, 1, 0, 0, 1, 1], derived=(Near,))  # columns 0-1 and 2-3
thirds = partition([0, 1, 1, 2, 0, 1, 1, 2])  # columns 0, 1-2 and 3
print("sentinels: MISSING_VALUE", MISSING_VALUE, "| TRUNCATED", TRUNCATED)
for local_mesh in halves:
    index, owned = local_mesh.global_index[Cell], local_mesh.owned[Cell]
    print(
        f"--- rank {local_mesh.rank}: global {index}, owned {owned}, {local_mesh.extents[Cell]} cells"
    )
    # rank 0, local cell 4 is the halo cell 2: its west neighbor is 1, its east one 3, truncated
    print("C2C  =", local_mesh[C2C].data.tolist())
    print("Near =", local_mesh[Near].data.tolist())
    print("regions:", {r.__name__: points for r, points in local_mesh.regions.items()})
print("the same mesh_id:", halves[0].mesh_id == halves[1].mesh_id)
try:
    partition([0, 0, 1, 1, 0, 0, 1, 1])[0][Near]  # not derived globally
except LookupError as error:
    print("LookupError:", error)
try:
    halves[0].rank = 1
except AttributeError as error:
    print("AttributeError:", error)

sentinels: MISSING_VALUE -2 | TRUNCATED -3
--- rank 0: global (0, 1, 4, 5, 2, 6), owned range(0, 4), 6 cells
C2C  = [[-2, 1, -2, 2], [0, 4, -2, 3], [-2, 3, 0, -2], [2, 5, 1, -2], [1, -3, -2, 5], [3, -3, 4, -2]]
Near = [[1, 2, -1, -1], [0, 4, 3, -1], [0, 3, -1, -1], [1, 2, 5, -1], [1, -3, 5, -1], [4, 3, -3, -1]]
regions: {'West': (0, 2), 'East': range(0, 0)}
--- rank 1: global (2, 3, 6, 7, 1, 5), owned range(0, 4), 6 cells
C2C  = [[4, 1, -2, 2], [0, -2, -2, 3], [5, 3, 0, -2], [2, -2, 1, -2], [-3, 0, -2, 5], [-3, 2, 4, -2]]
Near = [[4, 1, 2, -1], [0, 3, -1, -1], [0, 5, 3, -1], [1, 2, -1, -1], [-3, 0, 5, -1], [4, -3, 2, -1]]
regions: {'West': range(0, 0), 'East': (1, 3)}
the same mesh_id: True
LookupError: A local mesh does not derive the table of 'Near', from `C2C.T`: the halo rows of a transpose would miss the sources outside the local support. Derive it on the global mesh: `decompose(..., derived=[...])`.
AttributeError: A `LocalMesh` is immutable.


## Domains on a rank

| Concept | Kind | Meaning |
|---|---|---|
| `Owned(Cell)` | function | The domain of the points of `Cell` this rank owns, in the current local mesh (a `TypeError` outside one) |
| `region(West)` | function | The domain of the local points of `West`: halo replicas too, if their global points are in it |
| `intersect`, `difference` | functions | Combine domains, e.g. `difference(Owned(Cell), region(West))` |

The halo points cannot be named: results are requested on owned points or regions.

In [2]:
from concepts import Owned, region
from implementation import difference, intersect


def global_points(domain, local_mesh):
    return [local_mesh.global_index[Cell][i] for i in domain[Cell]]


for local_mesh in thirds:
    with local_mesh:
        owned, west = Owned(Cell), region(West)
        domains = [owned, west, intersect(owned, west), difference(owned, west)]
        print(f"rank {local_mesh.rank}:", *(global_points(d, local_mesh) for d in domains))
try:
    Owned(Cell)  # outside a local mesh
except TypeError as error:
    print("TypeError:", error)

rank 0: [0, 4] [0, 4] [0, 4] []
rank 1: [1, 2, 5, 6] [0, 4] [] [1, 2, 5, 6]
rank 2: [3, 7] [] [] [3, 7]
TypeError: No local mesh: use `with local_mesh:` to provide it.


## Operators on a rank

DSL operators run unchanged on a rank, in `with local_mesh:`, e.g. requested on `Owned(Cell)`:

- `scatter(field, local_mesh)` gives the values of a global field on the owned points, with local indices; with `with_halo=True`, on all the local points, as if exchanged;
- `gather_owned(fields, local_meshes)` puts the owned values of each rank back into a global field;
- a halo row with a `TRUNCATED` neighbor is not computable: forward, it is not in the domain; backward, a request on it raises a `ValueError` (beyond the local support).

The example: mass sharing. At each step, each cell passes `k` of its mass to each neighbor and receives `k` of each neighbor's mass, so the total is conserved.

In [3]:
from implementation import backward_domain, eager, field_operator, gather_owned, scatter


@field_operator()
def share(
    m: Field[Dims[Cell], float], one: Field[Dims[Cell], float], k: float
) -> Field[Dims[Cell], float]:
    received = sum_over(m(C2C.boundary(0.0)), axis=Neighbor[C2C])  # the neighbors' mass
    given = sum_over(one(C2C.boundary(0.0)), axis=Neighbor[C2C]) * m  # to each neighbor
    return m + k * (received - given)


m0 = NumPyArrayField({Cell: range(8)}, [8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
ones = NumPyArrayField({Cell: range(8)}, np.ones(8))
k = 0.125
with Mesh(plate):
    once = share(m0, ones, k)  # cell 0: 8 + 0.125 * (0 - 2 * 8) = 6; cells 1 and 4: 1
    print("globally =", once, "| total", sum(once.data.tolist()))

r0 = halves[0]  # owns the cells 0, 1, 4, 5 (local 0-3); its halo, 2 and 6 (local 4, 5)
with r0:
    # the halo rows are truncated: computed on Cell[0:4] only
    eager(share)(scatter(m0, r0, with_halo=True), scatter(ones, r0, with_halo=True), k)
    try:
        backward_domain({Cell: (4,)}, share.__gt_extents__["m"])
    except ValueError as error:
        print("ValueError:", error)

per_rank = []
for local_mesh in halves:
    with local_mesh:
        inputs = scatter(m0, local_mesh, with_halo=True), scatter(ones, local_mesh, with_halo=True)
        per_rank.append(share[Owned(Cell)](*inputs, k))
print("gathered =", gather_owned(per_rank, halves))

globally = Field(Cell[0:8]: [6.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]) | total 8.0
[eager] computed Field(Cell[0:4]: [6.0, 1.0, 1.0, 0.0])
ValueError: Cell[4:5] is beyond the local support: it has neighbors through 'C2C' outside it (decompose with more `hops`).
gathered = Field(Cell[0:8]: [6.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0])


## Local programs

`@local_program` marks a Python function that `run_local(program, local_meshes, args_per_rank)` runs on all the ranks at once, each in its `with local_mesh:` block, returning one result per rank. Its fields are *values*, valid on their domains: results are new fields. The collectives are called by all the ranks, in the same order:

- `exchange(f)`: `f` on all the local points, the halo too, with the values of their owners. `f` needs values on the owned points of its one distributed dimension; its other dimensions are kept whole (no `Pairs` in Phase 1).
- `global_sum(f, domain=...)`: the sum on owned points, over all the ranks, in the order of the global indices: the same on every rank, for any decomposition.

DSL operators, on any backend, run in a local program as anywhere. Below, `sharing` exchanges `one` once, then `m` before each step, and returns `m`, its total and the total in `East`.

In [4]:
from implementation import exchange, global_sum, local_program, run_local


@local_program
def sharing(m, one, k, steps, backend):
    one = exchange(one)  # its halo, once
    for _ in range(steps):
        m = exchange(m)  # the halo from the owners, before each step
        op = share if backend is None else backend(share)
        m = op[Owned(Cell)](m, one, k)
    east = global_sum(m, domain=intersect(Owned(Cell), region(East)))
    return m, global_sum(m, domain=Owned(Cell)), east


def run(program, local_meshes, m, steps, backend=None):
    """Run `program` from the global `m` and `ones` (each rank gets its owned values): the
    global field of the first results of the ranks, and the other results of each rank."""
    args = [(scatter(m, r), scatter(ones, r), k, steps, backend) for r in local_meshes]
    with contextlib.redirect_stdout(io.StringIO()):  # silence the backends, on all the threads
        results = run_local(program, local_meshes, args)
        return gather_owned([r[0] for r in results], local_meshes), [r[1:] for r in results]


partitions = {"single": single, "halves": halves, "thirds": thirds}
# cell 0: 8, 6, then 6 + (1 + 1 - 2 * 6) / 8 = 4.75, then 4.75 + (1.375 + 1.5 - 2 * 4.75) / 8
for name, local_meshes in partitions.items():
    m3, sums = run(sharing, local_meshes, m0, 3)
    print(f"{name}: {m3.data.tolist()} | total and east, per rank: {sums}")

single: [3.921875, 1.5, 0.25, 0.015625, 1.75, 0.515625, 0.046875, 0.0] | total and east, per rank: [(8.0, 0.015625)]
halves: [3.921875, 1.5, 0.25, 0.015625, 1.75, 0.515625, 0.046875, 0.0] | total and east, per rank: [(8.0, 0.015625), (8.0, 0.015625)]
thirds: [3.921875, 1.5, 0.25, 0.015625, 1.75, 0.515625, 0.046875, 0.0] | total and east, per rank: [(8.0, 0.015625), (8.0, 0.015625), (8.0, 0.015625)]


## The simulation condition

A local program with an exchange before each read of the halo computes the values of the global run, bitwise, for any decomposition and on any backend: below, 4 steps from `ms`, the square root of `c + 1` in cell `c`, on 1, 2 and 3 ranks.

In [5]:
import math

from implementation import column_major, compiled, lazy, lazy_backward

ms = NumPyArrayField({Cell: range(8)}, [math.sqrt(c + 1.0) for c in range(8)])
with Mesh(plate):
    expected = ms
    for _ in range(4):
        expected = share(expected, ones, k)
for backend in (None, eager, lazy, lazy_backward, column_major, compiled):
    same = [
        run(sharing, local_meshes, ms, 4, backend)[0].data.tolist() == expected.data.tolist()
        for local_meshes in partitions.values()
    ]
    print(f"{backend or 'embedded'}:", same)

embedded: [True, True, True]
eager: [True, True, True]
lazy: [True, True, True]


lazy_backward: [True, True, True]
column_major: [True, True, True]
compiled: [True, True, True]


## A missing exchange

Without `exchange(m)`, `m` has no halo: on rank 0, `m` is on its owned cells, `Cell[0:4]`, and `share` can only compute the owned cells whose neighbors are owned too, the local cells 0 and 2 (the global 0 and 4). The request on `Owned(Cell)` is a containment error, on every rank, not a hang: `run_local` raises the error of the first rank.

In [6]:
@local_program
def forgetful(m, one, k, steps, backend):
    one = exchange(one)
    for _ in range(steps):
        m = share[Owned(Cell)](m, one, k)  # no exchange of `m`
    return (m,)


try:
    run(forgetful, halves, m0, 1)
except ValueError as error:
    print("ValueError:", error)

ValueError: Cell[0:4] is not within the domain Cell{0, 2}.


## Reproducible sums

`global_sum` gathers the `(global index, value)` pairs of all the ranks and adds them one by one in global order: the same sum, bitwise, for any decomposition. Reproducible, not exact: below, the 1s added to `1e16` are lost in floating point, the same way for every decomposition, while the naive sums of the per-rank sums differ; `math.fsum` gives the exact sum.

In [7]:
v = NumPyArrayField({Cell: range(8)}, [1.0, 1e16, 1.0, -1e16, 1.0, 1.0, 1.0, 1.0])


@local_program
def sums(f):
    partial = 0.0
    for x in f[Owned(Cell)].data.tolist():  # the values of this rank, in local order
        partial += x
    return global_sum(f, domain=Owned(Cell)), partial


# in global order: 1 + 1e16 is 1e16 (the 1 is lost), + 1 is 1e16, - 1e16 is 0, then 4 * 1 = 4
for name, local_meshes in partitions.items():
    results = run_local(sums, local_meshes, [(scatter(v, r),) for r in local_meshes])
    naive = 0.0
    for _, partial in results:
        naive += partial
    print(f"{name}: global_sum {[s for s, _ in results]} | naive {naive}")
print("exact, math.fsum:", math.fsum(v.data.tolist()))

single: global_sum [4.0] | naive 4.0
halves: global_sum [4.0, 4.0] | naive 0.0
thirds: global_sum [4.0, 4.0, 4.0] | naive 2.0
exact, math.fsum: 6.0


## Fields are values; what DSL operators cannot do

- `exchange` and `global_sum` raise a `RuntimeError` outside a running local program.
- In a running local program, writes into fields, `f[...] = ...`, and `empty` or `empty_like` raise a `RuntimeError`: a field must not claim values it does not have, e.g. a buffer of the local points written on the owned ones only.
- DSL operators compute point by point: `FIELD_OPERATOR_RULES` rejects the `COLLECTIVES`, also through aliases, private names like `_write` (the private write path of the implementation), and writes into subscripts or attributes. A `global_sum` in a definition is rejected by the `domain=` rule first, so `exchange` is shown.
- `run_local` checks that the local meshes are parts of the same mesh: the same `mesh_id`.

In [8]:
from concepts import COLLECTIVES, current_mesh
from implementation import _write, numpy_field_allocator

try:
    exchange(m0)  # outside a local program
except RuntimeError as error:
    print("RuntimeError:", error)


@local_program
def write_halo(m):
    buffer = numpy_field_allocator.zeros({Cell: range(6)})  # the local cells, the halo too
    buffer[Owned(Cell)] = m  # would claim values on the halo it does not have
    return buffer


@local_program
def uninitialized(m):
    return numpy_field_allocator.empty({Cell: range(6)})


for program in (write_halo, uninitialized):
    try:
        run_local(program, halves, [(scatter(m0, r),) for r in halves])
    except RuntimeError as error:
        print("RuntimeError:", error)

halo = exchange  # an alias


def exchanging(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return exchange(m)


def aliased(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return halo(m)


def private(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    _write(m, ..., 0.0)
    return m


def written(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    m[...] = 0.0
    return m


for definition in (exchanging, aliased, private, written):
    try:
        field_operator()(definition)
    except TypeError as error:
        print("TypeError:", error)
print("COLLECTIVES:", sorted(COLLECTIVES))


@local_program
def rank():
    return current_mesh().rank


# another decomposition, another mesh_id (the ids count the decompositions made in this kernel)
other = partition([0, 0, 1, 1, 0, 0, 1, 1])
try:
    run_local(rank, [halves[0], other[1]], [(), ()])
except ValueError as error:
    print("ValueError:", error)

RuntimeError: `exchange` is a collective: call it in a local program, run on all the ranks by `run_local`.
RuntimeError: Fields are values in a local program: compute a new field instead of writing into Cell[0:6], e.g. `exchange(f)` for its halo.
RuntimeError: No `empty` fields in a local program: their values would not be valid on their domain, Cell[0:6]. Use `zeros` or `full`.
TypeError: 'exchanging' is not a valid field operator definition: uses the collective 'exchange'.
TypeError: 'aliased' is not a valid field operator definition: uses the collective 'exchange' (as 'halo').
TypeError: 'private' is not a valid field operator definition: uses the private name '_write'.
TypeError: 'written' is not a valid field operator definition: assignment to a subscript.
COLLECTIVES: ['exchange', 'global_sum']
ValueError: The local meshes are parts of different meshes, ['mesh-1', 'mesh-4'].
